In [6]:
import numpy as np
import pandas as pd
import os
import sys
import matlab
import matlab.engine
import pickle
%matplotlib widget
import matplotlib.pyplot as plt
from scipy.signal import detrend
from HRV_and_CPC_analysis_functions import load_ecg_data, load_sleep_data

In [21]:
# path = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_resampled_ECG/ECG_014.h5'
# data = load_ecg_data(path)
# data['I'].shape
# data['I_startdatetime']
# data['I'].shape


array([2018,   10,    8,   19,   25,    0,    0])

(41213778,)

In [2]:
matlab.engine

<module 'matlab.engine' from '/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/matlab/engine/__init__.py'>

In [3]:
np.pi

3.141592653589793

In [4]:
eng = matlab.engine.start_matlab()
eng

In [5]:
def spectrum_plot(var, ax, db=True, vmin=0.05, vmax=0.99):
    
    if db:
        var = 20*np.log10(var)
    vmin = np.quantile(var, vmin)
    vmax = np.quantile(var, vmax)
    im = ax.imshow(var, origin = 'lower', aspect = 'auto' , cmap = colormap, interpolation="none", vmin=vmin, vmax=vmax, extent=(t[0], t[-1], f[0], f[-1])) # ,vmin=q01, vmax=q09
    fig.colorbar(im, ax=ax)
    ax.set_yticks([0.01, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9])
    ax.set_ylabel('Hz')
    ax.set_ylim([0, 0.7])
    
    
def freq_domain_analysis():
    windowsize = 60*8.5    # in seconds
    increment = 13 # 26          # 26sec is 5% of 8.5min window

    nn_matlab = matlab.double(list(df.nn.values))
    band_matlab = matlab.double(list(df.band.values))

    C, phi, S12, S1, S2, t, f = eng.MultiTaperFreqDomainAnalysis_includingCPC(band_matlab, nn_matlab, windowsize, increment, fs, nargout=7)
    C = np.transpose(np.array(C))
    phi = np.transpose(np.array(phi))
    S12 = np.transpose(np.array(S12))
    S1 = np.transpose(np.array(S1))
    S2 = np.transpose(np.array(S2))
    t = np.array(t).flatten()
    f = np.array(f).flatten()
    CS = (np.conj(S12)*S12).real
    
    C_2 = C.copy()
    C_2[C_2<0.3] = 0
    CPC = np.multiply(C_2, CS)
    
    return [C, phi, CS, CPC, S1, S2, t, f]

In [7]:
fs = 2
len_data = fs*60*45
f_breathing = 0.1  # sec
breathing_w = f_breathing*2*np.pi
breathing_amplitude = 1

colormap="jet"

f_nn = 0.5 # sec
nn_w = f_nn*2*np.pi
nn_mean = 1
nn_amplitude = 1

samples = np.arange(len_data)
time = samples/fs

# step 1: two distinct signals
df = pd.DataFrame()
df['band'] = np.sin(breathing_w*time) * breathing_amplitude + np.random.normal(0, breathing_amplitude/50, size=(len_data,))
df['nn'] = nn_mean + np.sin(nn_w*time) * nn_amplitude + np.random.normal(0, nn_amplitude/50, size=(len_data,))


if 1:
    df['nn'].iloc[len_data//2:] +=  np.sin(time[len_data//2:]*2*np.pi*0.6)
    
if 0:
    # add common frequency:
    common_signal = np.sin(0.3*np.pi*2*time) * 1
    df['band'] = df['band'] + common_signal
    df['nn'] = df['nn'] + common_signal
if 0:
    # add common frequency, constant phase shift
    signal_a = np.sin(0.3*np.pi*2*time) * 1
    signal_a_phaseshift = np.sin(0.3*np.pi*2*(time+np.pi/4)) * 1
    df['band'] = df['band'] + signal_a
    df['nn'] = df['nn'] + signal_a_phaseshift
    
if 1:
    # add common frequency, varying phase shift
    signal_a = np.sin(0.3*np.pi*2*time) * 1
    phaseshift_vetor = np.arange(0, 1, 1/len(signal_a)) * np.pi*4 # vector that gradually goes from 0 to pi/4
    signal_a_phaseshift = np.sin(0.3*np.pi*2*(time+phaseshift_vetor)) * 0.25
    df['band'] = df['band'] + signal_a
    df['nn'] = df['nn'] + signal_a_phaseshift
    
fig, ax = plt.subplots(2,1,figsize=(10,2), sharex=True)
ax[0].plot(time, df.band)
ax[1].plot(time, df.nn)
ax[0].set_xlim([0,10])
plt.tight_layout()

df['band'] = detrend(df['band'])
df['nn'] = detrend(df['nn'])
[C, phi, CS, CPC, S1, S2, t, f] = freq_domain_analysis()

fig,(ax1, ax2, ax3, ax4, ax5) = plt.subplots(5,1, figsize=(10,8))
spectrum_plot(S1, ax1, db=True, vmax=0.99)
spectrum_plot(S2, ax2, db=True, vmax=0.99)
spectrum_plot(C, ax3, db=False, vmin=0.5, vmax=0.99)
spectrum_plot(CS, ax4, db=True, vmax=0.99)
spectrum_plot(CPC, ax5, db=True, vmin=0.6, vmax=0.99)
plt.suptitle('Resp., NN, Coherence, Cross Spectrum, CPC')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:4: RuntimeWarning: divide by zero encountered in log10
  after removing the cwd from sys.path.


Text(0.5, 0.98, 'Resp., NN, Coherence, Cross Spectrum, CPC')

In [174]:
plt.figure()
plt.hist(C.flatten())

C:\Users\wg984\AppData\Local\Continuum\anaconda3\lib\site-packages\ipykernel_launcher.py:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  """Entry point for launching an IPython kernel.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

(array([ 1836.,  5234.,  8347., 11021., 13134., 13265., 12276., 10057.,
         7201.,  3312.]),
 array([1.87377129e-04, 9.93221660e-02, 1.98456955e-01, 2.97591744e-01,
        3.96726533e-01, 4.95861321e-01, 5.94996110e-01, 6.94130899e-01,
        7.93265688e-01, 8.92400477e-01, 9.91535266e-01]),
 <a list of 10 Patch objects>)

In [95]:
C.min()

0.002330519805157284

In [96]:
CS.min()

1.209825244199806e-13

In [98]:
CPC.min()

3.251128194413444e-16

In [97]:
CS.max()

10.41136409259758

In [131]:
CPC.max()

6.34063039123855

In [40]:
[C, phi, S12, S1, S2, t, f] = freq_domain_analysis()

fig,(ax1, ax2, ax3) = plt.subplots(3,1)
spectrum_plot(S1, ax1, db=True)
spectrum_plot(S2, ax2, db=True)
spectrum_plot(C, ax3, db=False, vmin=0.5)


C:\Users\wg984\AppData\Local\Continuum\anaconda3\lib\site-packages\ipykernel_launcher.py:3: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  This is separate from the ipykernel package so we can avoid doing imports until


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [130]:
for i in range(100):
    plt.close()